In [1]:
# Imported Sionna and the functions we will use
# Imported the NumPy library for mathematical operations
import sionna.rt
import mitsuba as mi
from sionna.rt import (
    PlanarArray,
    Transmitter,
    Receiver,
    load_scene,
    PathSolver,
    SceneObject,
    ITURadioMaterial
    )
import numpy as np

jitc_llvm_init(): LLVM API initialization failed ..


In [2]:
# Called our environment and loaded the scene
scene = load_scene()

In [3]:
# Set our frequency to 3.5 GHz
scene.frequency = 3.5e9

In [4]:
# Using the PlanarArray function, we defined the number of Tx antenna elements, to be used as our base station, antenna properties and orientation

scene.tx_array= PlanarArray(
    num_rows=1, 
    num_cols=1,
    pattern="iso", 
    polarization="V",)

In [5]:
# We defined the properties of our receiving antenna
scene.rx_array= PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V",
)

In [6]:
# Named our transmitter Base Station 1 and defined its position
bs = Transmitter(
    name="BS1",
    position=[0, 0, 0]
)

In [7]:
# Added our base station to the scene
scene.add(bs)

In [8]:
# Set r = 20 meters as a constant value
r = 20

In [9]:
# Named our first receiver and set its position at a distance of r from the base station
rx1 = Receiver(
    name="RX1",
    position=[r, 0, 0]
)

In [10]:
# Named our second receiver and set its position at a distance of 2*r from the base station# Named our second receiver and set its position at a distance of 2*r from the base station
rx2 = Receiver(
    name="RX2",
    position=[2*r, 0, 0]
)

In [11]:
# Named our third receiver and set its position at a distance of 3*r from the base station
rx3 = Receiver(
    name="RX3",
    position=[3*r, 0, 0]
)

In [12]:
# Named our fourth receiver and set its position at a distance of 4*r from the base station
rx4 = Receiver(
    name="RX4",
    position=[4*r, 0, 0]
)

In [13]:
# Added our receivers to the scene
scene.add(rx1)
scene.add(rx2)
scene.add(rx3)
scene.add(rx4)

In [14]:
# Updated our receiver's position to 10*r using .position and added it to the scene again
rx4 = scene.get("RX4")
rx4.position = [10*r, 0, 0]

In [15]:
# Printed the position of each receiver to make sure their positions were correct
for name in ["RX1", "RX2", "RX3", "RX4"]:
    rx = scene.get(name)
    print(name, np.array(rx.position))

RX1 [[20.]
 [ 0.]
 [ 0.]]
RX2 [[40.]
 [ 0.]
 [ 0.]]
RX3 [[60.]
 [ 0.]
 [ 0.]]
RX4 [[200.]
 [  0.]
 [  0.]]


In [16]:
# Defined the constant speed of light, wavelength, and base station power
# Calculated the free space path loss values for each receiver
# path gain = - path loss
# Calculated the received signal strength (RSS) for each receiver
c= 299792458.0
wavelength = c/ scene.frequency
bs_power_dbm = 30
fspl1 = 20*np.log10(4*np.pi*rx1.position[0]/wavelength)
fspl2 = 20*np.log10(4*np.pi*rx2.position[0]/wavelength)
fspl3 = 20*np.log10(4*np.pi*rx3.position[0]/wavelength)
fspl4 = 20*np.log10(4*np.pi*rx4.position[0]/wavelength)

rss1 = bs_power_dbm - fspl1
rss2 = bs_power_dbm - fspl2
rss3 = bs_power_dbm - fspl3
rss4 = bs_power_dbm - fspl4

print(f"RSS at RX1: {rss1} dBm")
print(f"RSS at RX2: {rss2} dBm")
print(f"RSS at RX3: {rss3} dBm")
print(f"RSS at RX4: {rss4} dBm")

print(f"FSPL at RX1: {fspl1} dB")
print(f"FSPL at RX2: {fspl2} dB")
print(f"FSPL at RX3: {fspl3} dB")
print(f"FSPL at RX4: {fspl4} dB")

# Convert received power from dBm to nanowatts for comparing
p_rx1 = 10**((rss1 - 30) / 10) * 1e9
p_rx2 = 10**((rss2 - 30) / 10) * 1e9
p_rx3 = 10**((rss3 - 30) / 10) * 1e9
p_rx4 = 10**((rss4 - 30) / 10) * 1e9

print(f"RX1: {p_rx1.item():.2f} nW")
print(f"RX2: {p_rx2.item():.2f} nW")
print(f"RX3: {p_rx3.item():.2f} nW")
print(f"RX4: {p_rx4.item():.2f} nW")

print(f"RX2 / RX1: {((p_rx2 / p_rx1).item()):.2f}")

RSS at RX1: [-39.349747] dBm
RSS at RX2: [-45.370346] dBm
RSS at RX3: [-48.892166] dBm
RSS at RX4: [-59.349747] dBm
FSPL at RX1: [69.34975] dB
FSPL at RX2: [75.370346] dB
FSPL at RX3: [78.892166] dB
FSPL at RX4: [89.34975] dB
RX1: 116.15 nW
RX2: 29.04 nW
RX3: 12.91 nW
RX4: 1.16 nW
RX2 / RX1: 0.25


In [17]:
building = SceneObject(
    fname="building1.ply",
    name="building1",
    radio_material=ITURadioMaterial(
        name="concrete",
        itu_type="concrete"
    )
)

In [18]:
solver = PathSolver()

paths = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    diffraction=True
)

In [19]:
#obje kapatırsa  rss ve path gain değerleri değişimi
# Get channel coefficients
a_real, a_imag = paths.a
a = np.array(a_real) + 1j * np.array(a_imag)

# Calculate path gain
h = np.sum(a, axis=-1)
path_gain = np.abs(h)**2
path_gain_db = 10 * np.log10(path_gain + 1e-30)

# Calculate RSS
bs_power_dbm = 30
rss = bs_power_dbm + path_gain_db

# Print results
for i in range(4):
    print(f"RX{i+1} Path Gain: {path_gain_db[i].item():.2f} dB")
    print(f"RX{i+1} RSS: {rss[i].item():.2f} dBm")

RX1 Path Gain: -69.35 dB
RX1 RSS: -39.35 dBm
RX2 Path Gain: -75.37 dB
RX2 RSS: -45.37 dBm
RX3 Path Gain: -78.89 dB
RX3 RSS: -48.89 dBm
RX4 Path Gain: -89.35 dB
RX4 RSS: -59.35 dBm


In [20]:
scene.add(building)

In [21]:
# Displayed the scene preview
scene.preview()